In [7]:

import os
from typing import Optional

import ms3
import pandas as pd

from notebooks.utils import DivMaker, onset2beat

DLC_PATH = ms3.resolve_dir("~/distant_listening_corpus")

In [8]:
def filter_corpus(corpus):
    corpus.view.include("facets", "scores")#, "expanded")
    #corpus.disambiguate_facet("expanded")
    corpus.disambiguate_facet("scores")
    corpus.view.pieces_with_incomplete_facets = False
    
def get_ms3_corpus(corpus_path):
    corpus = ms3.Corpus(corpus_path)
    filter_corpus(corpus)
    return corpus

for subcorpus_dir in os.listdir(DLC_PATH):
    subcorpus_path = os.path.join(DLC_PATH, subcorpus_dir)
    if os.path.isfile(subcorpus_path): continue
    corpus = get_ms3_corpus(subcorpus_path)
    break
    
corpus

[default|all]
Corpus 'kozeluh_sonatas'
------------------------
Location: /home/laser/distant_listening_corpus/kozeluh_sonatas
View: This view is called 'default'. It 
	- excludes pieces that are not contained in the metadata,
	- filters out file extensions requiring conversion (such as .xml),
	- excludes review files and folders,
	- includes only facets containing 'scores', and
	- excludes pieces that do not have at least one file per selected facet (scores).

All 49 pieces are listed in 'metadata.tsv':

             scores
           detected
09op08no1a        1
09op08no1b        1
10op08no2a        1
10op08no2b        1
10op08no2c        1
11op10no1a        1
14op13no2a        1
14op13no2b        1
14op13no2c        1
15op13no3a        1
15op13no3b        1
15op13no3c        1
16op15no1a        1
16op15no1b        1
16op15no1c        1
17op15no2a        1
17op15no2b        1
17op15no2c        1
19op17no1a        1
19op17no1b        1
19op17no1c        1
20op17no2a        1
20op17no2

In [9]:
corpus = get_ms3_corpus("~/distant_listening_corpus/beethoven_piano_sonatas")

for _, piece in corpus.iter_pieces():
    break
    
for fileinfo, facets in piece.iter_extracted_facets(
        ("notes", "expanded"),
        force=True,
        unfold=True,
        interval_index=False
):
    break
    
notes, labels = facets["notes"], facets["expanded"]
notes["is_onset"] = (notes.tied.fillna(1) == 1)
labels["is_onset"] = True
display(notes.head(3))
labels.head(3)

,mc,mn,mc_playthrough,mn_playthrough,quarterbeats_playthrough,quarterbeats_all_endings,duration_qb,mc_onset,mn_onset,timesig,...,gracenote,nominal_duration,scalar,tied,tpc,midi,name,octave,chord_id,is_onset
0,1,0,1,0a,0,0,1.0,0,3/4,2/2,...,NaN,1/4,1,<NA>,0,60,C4,4,0,True
1,2,1,2,1a,1,1,1.0,0,0,2/2,...,NaN,1/4,1,<NA>,-1,65,F4,4,1,True
2,2,1,2,1a,2,2,1.0,1/4,1/4,2/2,...,NaN,1/4,1,<NA>,-4,68,Ab4,4,2,True


,mc,mn,mc_playthrough,mn_playthrough,quarterbeats_playthrough,quarterbeats_all_endings,duration_qb,mc_onset,mn_onset,timesig,...,cadence,phraseend,chord_type,globalkey_is_minor,localkey_is_minor,chord_tones,added_tones,root,bass_note,is_onset
0,1,0,1,0a,0,0,9.0,0,3/4,2/2,...,NaN,{,m,True,True,"(0, -3, 1)",(),0,0,True
1,4,3,4,3a,9,9,8.0,0,0,2/2,...,NaN,NaN,Mm7,True,True,"(5, 2, -1, 1)",(),1,5,True
2,6,5,6,5a,17,17,4.0,0,0,2/2,...,NaN,NaN,m,True,True,"(0, -3, 1)",(),0,0,True


In [10]:
merged = pd.merge(
    left = notes, 
    right = labels, 
    on = "quarterbeats_playthrough",
    how = "outer",
    suffixes = ("", "_label"),
    indicator=True
)
merged

,mc,mn,mc_playthrough,mn_playthrough,quarterbeats_playthrough,quarterbeats_all_endings,duration_qb,mc_onset,mn_onset,timesig,...,phraseend,chord_type,globalkey_is_minor,localkey_is_minor,chord_tones,added_tones,root,bass_note,is_onset_label,_merge
0,1,0,1,0a,0,0,1.0,0,3/4,2/2,...,{,m,True,True,"(0, -3, 1)",(),0,0,True,both
1,2,1,2,1a,1,1,1.0,0,0,2/2,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,left_only
2,2,1,2,1a,2,2,1.0,1/4,1/4,2/2,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,left_only
3,2,1,2,1a,3,3,1.0,1/2,1/2,2/2,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,left_only
4,2,1,2,1a,4,4,1.0,3/4,3/4,2/2,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3381,154,152,308,152b,1213,605,1.0,0,0,2/2,...,},m,True,True,"(0, -3, 1)",(),0,0,True,both
3382,154,152,308,152b,1213,605,1.0,0,0,2/2,...,},m,True,True,"(0, -3, 1)",(),0,0,True,both
3383,154,152,308,152b,1213,605,1.0,0,0,2/2,...,},m,True,True,"(0, -3, 1)",(),0,0,True,both
3384,154,152,308,152b,1213,605,1.0,0,0,2/2,...,},m,True,True,"(0, -3, 1)",(),0,0,True,both


In [11]:
KEEP_ORIGINAL_COLUMNS = ["mc", "mn", "mc_playthrough", "mn_playthrough", "quarterbeats_playthrough", "duration", "staff", "voice"]
RENAME_ORIGINAL_COLUMNS = dict(
        midi = "pitch",
    is_onset = "note_is_onset",
    is_onset_label = "harmony_is_onset",
    )
COLUMN_ORDER = ["onset_div", "duration_div", "pitch", "step", "alter", "ts_beats", "ts_beat_type", "staff", "voice"]

def make_pitch_array(notes: pd.DataFrame, beat_decimals: Optional[int] = 3) -> pd.DataFrame:
    
    div_maker = DivMaker(
        onsets = notes.quarterbeats_playthrough, 
        durations =notes.duration * 4 # normally duration_qb but due to a bug these are currently floats
    )
    onset_div, duration_div = div_maker[("onsets", "durations")]
    
    keep_original_columns = [col for col in KEEP_ORIGINAL_COLUMNS if col in notes.columns]
    original_columns = notes[keep_original_columns]
    
    rename_original_columns = {k: v for k, v in RENAME_ORIGINAL_COLUMNS.items() if k in notes.columns}
    renamed_columns = notes[list(rename_original_columns.keys())].rename(columns=rename_original_columns)
    
    new_dataframes = []  # will be added as-is
    new_columns = dict() # will be renamed based on the keys
    
    # specific pitch
    specific_pitch = notes.name.str.extract(r"^(?P<step>[A-G])(?P<accidentals>b*|#*)(?P<octave>\d)$")
    new_dataframes.append(specific_pitch[["step", "octave"]])
    new_columns["alter"] = specific_pitch.accidentals.str.count("#") - specific_pitch.accidentals.str.count("b")
    
    # time signatures & beats
    new_dataframes.append(
        notes.timesig.str.extract(r"^(?P<ts_beats>\d+)/(?P<ts_beat_type>\d+)$")
    )
    new_columns["beat"] = ms3.transform(merged, onset2beat, ["mn_onset", "timesig"], round_to=beat_decimals)

    result = pd.concat(
        [
            pd.DataFrame(
                dict(
                    onset_div=onset_div,
                    duration_div=duration_div
                )
            ),
            pd.concat(new_columns, axis=1),
            renamed_columns,
            original_columns
        ] + new_dataframes,
        axis=1
    )
    column_order = [col for col in COLUMN_ORDER if col in result.columns]
    column_order += [col for col in result.columns if col not in column_order]
    return result[column_order]

pitch_array = make_pitch_array(merged)
pitch_array.to_csv("beethoven1.tsv", sep="\t", index=False)
pitch_array

,onset_div,duration_div,pitch,step,alter,ts_beats,ts_beat_type,staff,voice,beat,note_is_onset,harmony_is_onset,mc,mn,mc_playthrough,mn_playthrough,quarterbeats_playthrough,duration,octave
0,0,6,60,C,0,2,2,1,1,2.5,True,True,1,0,1,0a,0,1/4,4
1,6,6,65,F,0,2,2,1,1,1.0,True,NaN,2,1,2,1a,1,1/4,4
2,12,6,68,A,-1,2,2,1,1,1.5,True,NaN,2,1,2,1a,2,1/4,4
3,18,6,72,C,0,2,2,1,1,2.0,True,NaN,2,1,2,1a,3,1/4,5
4,24,6,77,F,0,2,2,1,1,2.5,True,NaN,2,1,2,1a,4,1/4,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3381,7278,6,53,F,0,2,2,2,1,1.0,True,True,154,152,308,152b,1213,1/4,3
3382,7278,6,65,F,0,2,2,1,1,1.0,True,True,154,152,308,152b,1213,1/4,4
3383,7278,6,68,A,-1,2,2,1,1,1.0,True,True,154,152,308,152b,1213,1/4,4
3384,7278,6,72,C,0,2,2,1,1,1.0,True,True,154,152,308,152b,1213,1/4,5
